In [1]:
import io
import gradio as gr
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

matplotlib.use("Agg")



d:\Project\Git_Hub\Ideas\Handwritten-Digit-Recognition-using-ANN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==========================================
# 1. PURE NUMPY NEURAL NETWORK (FROM SCRATCH)
# ==========================================
print("Training Neural Network from scratch...")
digits = load_digits()
X, y = digits.data / 16.0, digits.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)
Y_train_oh = np.eye(10)[y_train]

np.random.seed(42)


Training Neural Network from scratch...


In [3]:
# Architecture: 64 Input (8x8 pixels) -> 32 Hidden (ReLU) -> 10 Output (Softmax)
W1 = np.random.randn(64, 32) * np.sqrt(2.0 / 64)
b1 = np.zeros(32)
W2 = np.random.randn(32, 10) * np.sqrt(2.0 / 32)
b2 = np.zeros(10)

lr = 0.25
loss_history = []
acc_history = []

for epoch in range(600):
    # Forward pass
    z1 = X_train @ W1 + b1
    h = np.maximum(0, z1)  # ReLU activation
    z2 = h @ W2 + b2
    exp_z2 = np.exp(z2 - np.max(z2, axis=1, keepdims=True))
    p = exp_z2 / np.sum(exp_z2, axis=1, keepdims=True)  # Softmax

    # Cross-entropy loss & accuracy
    loss = -np.mean(np.sum(Y_train_oh * np.log(p + 1e-9), axis=1))
    acc = np.mean(np.argmax(p, axis=1) == y_train)
    loss_history.append(loss)
    acc_history.append(acc)

    # Backpropagation
    d2 = (p - Y_train_oh) / len(X_train)
    d1 = (d2 @ W2.T) * (z1 > 0)

    W2 -= lr * (h.T @ d2)
    b2 -= lr * np.sum(d2, axis=0)
    W1 -= lr * (X_train.T @ d1)
    b1 -= lr * np.sum(d1, axis=0)

# Evaluate test accuracy
h_test = np.maximum(0, X_test @ W1 + b1)
p_test = np.exp(h_test @ W2 + b2)
p_test /= np.sum(p_test, axis=1, keepdims=True)
test_acc = np.mean(np.argmax(p_test, axis=1) == y_test) * 100
print(f"Training Complete! Test Accuracy: {test_acc:.2f}%")




Training Complete! Test Accuracy: 97.04%


In [4]:
# ==========================================
# 2. PREPROCESSING & FORWARD PASS
# ==========================================
def preprocess_drawing(sketch_data):
    if sketch_data is None or sketch_data.get("composite") is None:
        return None

    # Convert canvas to grayscale and invert (black bg, white ink)
    img = Image.fromarray(sketch_data["composite"]).convert("L")
    img = ImageOps.invert(img)

    bbox = img.getbbox()
    if bbox is None:
        return None

    # Crop around the drawn digit and center inside an 8x8 grid (matching sklearn digits)
    cropped = img.crop(bbox)
    cropped.thumbnail((6, 6), Image.Resampling.LANCZOS)

    grid_8x8 = Image.new("L", (8, 8), 0)
    paste_x = (8 - cropped.width) // 2
    paste_y = (8 - cropped.height) // 2
    grid_8x8.paste(cropped, (paste_x, paste_y))

    arr = np.array(grid_8x8, dtype=np.float32)
    if arr.max() > 0:
        arr = arr / arr.max()  # Normalize to [0, 1]
    return arr


def forward_pass(input_8x8):
    x_vec = input_8x8.flatten().reshape(1, 64)
    h_act = np.maximum(0, x_vec @ W1 + b1)[0]  # 32 hidden activations
    z2 = h_act @ W2 + b2
    exp_z2 = np.exp(z2 - np.max(z2))
    probs = exp_z2 / np.sum(exp_z2)  # 10 output probabilities
    return x_vec[0], h_act, probs




In [7]:
# ==========================================
# 3. NEURAL NETWORK VISUALIZER (DARK THEME)
# ==========================================
def render_network_visual(input_8x8, x_vec, h_act, probs):
    pred_digit = int(np.argmax(probs))
    confidence = probs[pred_digit] * 100

    fig = plt.figure(figsize=(11, 5.4), facecolor="#141413")
    ax = fig.add_axes([0, 0, 1, 1], facecolor="#141413")
    ax.set_xlim(0, 11)
    ax.set_ylim(0, 5.4)
    ax.axis("off")

    # Header stats (like the reel)
    ax.text(
        0.4,
        5.05,
        f"input: 8x8 pixels (64 nodes)",
        color="#a3a39e",
        fontsize=9,
        fontfamily="monospace",
    )
    ax.text(
        6.5,
        5.05,
        f"loss: {loss_history[-1]:.3f}  |  test acc: {test_acc:.1f}%",
        color="#e07a5f",
        fontsize=9,
        fontfamily="monospace",
    )
    ax.text(
        9.1,
        5.05,
        f"pred: {pred_digit} ({confidence:.0f}%)",
        color="#4ade80",
        fontsize=10,
        fontweight="bold",
        fontfamily="monospace",
    )

    # 1. Left Box: Pixelated 8x8 Input Image
    ax_img = fig.add_axes([0.035, 0.36, 0.16, 0.34])
    ax_img.imshow(input_8x8, cmap="gray", vmin=0, vmax=1)
    ax_img.set_xticks([])
    ax_img.set_yticks([])
    for spine in ax_img.spines.values():
        spine.set_edgecolor("#444440")
        spine.set_linewidth(1.5)
    ax.text(
        1.25,
        1.65,
        "Preprocessed\n8x8 Input Grid",
        color="#d4d4d0",
        fontsize=8.5,
        ha="center",
        fontfamily="monospace",
    )# Node coordinates
    # Show 24 representative input nodes out of 64 for clean visual spacing
    in_indices = np.linspace(0, 63, 24, dtype=int)
    in_y = np.linspace(4.5, 0.7, len(in_indices))
    in_x = np.full_like(in_y, 3.1)

    hid_y = np.linspace(4.6, 0.6, 32)
    hid_x = np.full_like(hid_y, 5.6)

    out_y = np.linspace(4.4, 0.8, 10)
    out_x = np.full_like(out_y, 8.0)

    # Normalize hidden activations for brightness
    h_norm = h_act / (np.max(h_act) + 1e-6)

    # 2. Draw Connections: Input -> Hidden (highlight active pathways)
    for idx_pos, feat_idx in enumerate(in_indices):
        val = x_vec[feat_idx]
        if val < 0.15:
            continue
        for h_i in range(32):
            if h_norm[h_i] < 0.2:
                continue
            weight = W1[feat_idx, h_i]
            signal = val * h_norm[h_i] * abs(weight)
            if signal > 0.12:
                color = "#f59e0b" if weight > 0 else "#38bdf8"
                alpha = min(0.75, signal * 1.4)
                ax.plot(
                    [in_x[idx_pos], hid_x[h_i]],
                    [in_y[idx_pos], hid_y[h_i]],
                    color=color,
                    alpha=alpha,
                    linewidth=0.8,
                    zorder=1,
                )

    # 3. Draw Connections: Hidden -> Output (highlight firing signals)
    for h_i in range(32):
        if h_norm[h_i] < 0.15:
            continue
        for o_i in range(10):
            weight = W2[h_i, o_i]
            signal = h_norm[h_i] * abs(weight)
            if signal > 0.25:
                is_winner = o_i == pred_digit
                color = "#4ade80" if is_winner else "#d97706"
                alpha = min(0.85, signal * 0.6) if is_winner else 0.18
                lw = 1.2 if is_winner else 0.6
                ax.plot(
                    [hid_x[h_i], out_x[o_i]],
                    [hid_y[h_i], out_y[o_i]],
                    color=color,
                    alpha=alpha,
                    linewidth=lw,
                    zorder=1,
                )

    # 4. Draw Input Nodes
    for idx_pos, feat_idx in enumerate(in_indices):
        intensity = x_vec[feat_idx]
        ax.scatter(
            in_x[idx_pos],
            in_y[idx_pos],
            s=45,
            color=plt.cm.gray(intensity),
            edgecolors="#787872",
            linewidth=0.8,
            zorder=3,
        )

    # 5. Draw Hidden Layer Nodes (32 ReLU)
    for h_i in range(32):
        act = h_norm[h_i]
        node_color = plt.cm.YlOrBr(0.15 + 0.85 * act) if act > 0.05 else "#262624"
        ax.scatter(
            hid_x[h_i],
            hid_y[h_i],
            s=40,
            color=node_color,
            edgecolors="#f59e0b" if act > 0.3 else "#575752",
            linewidth=0.8,
            zorder=3,
        )

    # 6. Draw Output Layer Nodes (0-9) + Live Probability Bars
    for o_i in range(10):
        prob = probs[o_i]
        is_winner = o_i == pred_digit
        node_color = "#f59e0b" if is_winner else "#262624"
        edge_color = "#fef08a" if is_winner else "#787872"

        ax.scatter(
            out_x[o_i],
            out_y[o_i],
            s=130,
            color=node_color,
            edgecolors=edge_color,
            linewidth=1.3,
            zorder=4,
        )
        ax.text(
            out_x[o_i],
            out_y[o_i],
            str(o_i),
            color="#141413" if is_winner else "#e5e5e0",
            fontsize=8.5,
            fontweight="bold",
            ha="center",
            va="center",
            zorder=5,
            fontfamily="monospace",
        )

        # Probability bar next to each digit
        bar_x_start = 8.35
        max_bar_width = 1.8
        ax.barh(
            out_y[o_i],
            max_bar_width,
            left=bar_x_start,
            height=0.18,
            color="#262624",
            zorder=2,
        )
        ax.barh(
            out_y[o_i],
            max_bar_width * prob,
            left=bar_x_start,
            height=0.18,
            color="#4ade80" if is_winner else "#78716c",
            zorder=3,
        )
        ax.text(
            bar_x_start + max_bar_width + 0.12,
            out_y[o_i],
            f"{prob*100:4.1f}%",
            color="#4ade80" if is_winner else "#a3a39e",
            fontsize=8,
            va="center",
            fontfamily="monospace",
        )

    # Layer labels at bottom
    ax.text(
        3.1,
        0.2,
        "in · 64",
        color="#a3a39e",
        fontsize=9,
        ha="center",
        fontfamily="monospace",
    )
    ax.text(
        5.6,
        0.2,
        "hidden · 32 · ReLU",
        color="#f59e0b",
        fontsize=9,
        ha="center",
        fontfamily="monospace",
    )
    ax.text(
        8.0,
        0.2,
        "out · 10 · Softmax",
        color="#4ade80",
        fontsize=9,
        ha="center",
        fontfamily="monospace",
    )

    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=150, facecolor="#141413")
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf)




    

In [10]:
# ==========================================
# 4. INTERACTIVE GRADIO APP
# ==========================================
def analyze_handwriting(sketch_data):
    input_8x8 = preprocess_drawing(sketch_data)
    if input_8x8 is None:
        # Show sample digit '2' from dataset on startup/clear
        input_8x8 = X_test[y_test == 2][0].reshape(8, 8)
    x_vec, h_act, probs = forward_pass(input_8x8)
    viz_img = render_network_visual(input_8x8, x_vec, h_act, probs)
    pred = int(np.argmax(probs))
    return viz_img, f"Predicted Digit: {pred}  (Confidence: {probs[pred]*100:.2f}%)"


with gr.Blocks(theme=gr.themes.Base(), title="ANN Live Neural Firing Visualizer") as demo:
    gr.Markdown("""
        #  Live Artificial Neural Network Visualizer (From Scratch in NumPy)
        **Zeal College of Engineering & Research | Dept. of AI & DS | CIE-II Activity**
        Draw a single digit (**0–9**) on the left panel. Watch how the image is compressed into an **8×8 pixel grid (64 inputs)**, triggers **32 ReLU hidden neurons**, and computes **Softmax probabilities** in real time.
        """)

    with gr.Row():
        with gr.Column(scale=1):
            sketchpad = gr.Sketchpad(
                label="Write a Digit Here (0–9)",
                type="numpy",
                height=280,
                width=280,
                brush=gr.Brush(default_size=18, colors=["#000000"]),
            )
            with gr.Row():
                clear_btn = gr.ClearButton([sketchpad], value="Clear")
                run_btn = gr.Button("Detect & Visualize", variant="primary")
            result_banner = gr.Textbox(
                label="Output Classification",
                value="Draw a digit to see live neural firing!",
                interactive=False,
            )

        with gr.Column(scale=3):
            network_plot = gr.Image(
                label="Real-Time Forward Propagation (64 → 32 ReLU → 10 Softmax)",
                type="pil",
            )

    # Load initial sample visualization on page open
    demo.load(fn=analyze_handwriting, inputs=sketchpad, outputs=[network_plot, result_banner])

    # Update live whenever user draws or clicks Detect
    run_btn.click(fn=analyze_handwriting, inputs=sketchpad, outputs=[network_plot, result_banner])
    sketchpad.change(fn=analyze_handwriting, inputs=sketchpad, outputs=[network_plot, result_banner])

if __name__ == "__main__":
    demo.launch()

C:\Users\Dell\AppData\Local\Temp\ipykernel_4200\3638775397.py:15: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Base(), title="ANN Live Neural Firing Visualizer") as demo:


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
